# Lab 2: Embeddings and retrieval

## Overview

An embedding turns a piece of text into a list of numbers that captures its meaning. Texts about the same thing end up with similar numbers, which makes embeddings the standard tool for semantic search and for giving a chat model facts it was never trained on.

The hackathon endpoint serves the `Qwen3-Embedding-0.6B` model. Each text becomes a vector of 1,024 numbers. In this lab you embed a small set of event notes, search them by meaning, and then feed the best matches to the chat model so it answers from your notes instead of guessing.

## Learning objectives

By the end of this lab you will be able to:

- Request an embedding and read its shape
- Embed many texts within the 16-input limit per request
- Rank texts by cosine similarity to a query
- Combine retrieval with chat completions for grounded answers
- Know the size limits that apply to embedding requests

## Prerequisites

- Lab 1 completed, including the `.env` file
- About 20 minutes

## Step 1: Set up your environment

In [ ]:
# Installs the packages this lab needs. Safe to rerun.
%pip install -q "openai>=3,<4" python-dotenv numpy

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Reads OPENAI_BASE_URL and OPENAI_API_KEY from the .env file in the repository root.
# override=True makes the file win over any OPENAI_API_KEY already set in your shell.
load_dotenv(Path("..") / ".env", override=True)
load_dotenv(override=True)

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]
print("Endpoint:", base_url)
print("Key:     ", api_key[:6] + "..." + api_key[-4:])

client = OpenAI()  # The SDK reads the same two variables, so no arguments are needed.

## The call in one look

`POST {OPENAI_BASE_URL}/embeddings` with header `Authorization: Bearer <team key>` and a JSON body.

| Field | Required | Values and defaults |
|---|---|---|
| `model` | yes | `"hackathon-embedding"` |
| `input` | yes | one string, or a list of up to 16 strings |
| `encoding_format` | no | `"float"` or `"base64"`; the SDK picks base64 and decodes it for you |

The response has `data[i].embedding`, a list of 1,024 floats per input in the same order, and `usage.prompt_tokens`. The minimal call:

In [ ]:
response = client.embeddings.create(
    model="hackathon-embedding",
    input=["first text", "second text"],
)
print(len(response.data), "vectors of", len(response.data[0].embedding), "numbers")

## Step 2: Embed one text

The request is the same shape as the OpenAI embeddings API. The model alias is `hackathon-embedding`.

In [ ]:
import numpy as np

response = client.embeddings.create(
    model="hackathon-embedding",
    input="Akamai Cloud runs GPU instances in Seattle.",
)
vector = response.data[0].embedding

print("Model:      ", response.model)
print("Dimensions: ", len(vector))
print("First five: ", [round(v, 4) for v in vector[:5]])
print("Usage:      ", response.usage)

Embeddings count toward your team's 60 requests per minute but not toward the chat token quota.

## Step 3: Embed many texts

You can pass a list instead of a string, but the server accepts at most 16 inputs per request. Here is what happens with 17.

In [ ]:
import openai

try:
    client.embeddings.create(model="hackathon-embedding", input=[f"sentence {i}" for i in range(17)])
except openai.APIStatusError as e:
    print("Status:", e.status_code)
    print("Body:  ", e.response.text[:200])

A small helper that splits any list into batches of 16 keeps the rest of your code simple, and retries when the server answers `429`. It returns a NumPy array with one row per input, in the original order.

In [ ]:
import random
import time

BATCH = 16  # Server-side maximum inputs per request.

def embed(texts, attempts=5):
    """Embed any number of strings, 16 per request, preserving order. Retries on 429."""
    vectors = []
    for start in range(0, len(texts), BATCH):
        batch = texts[start:start + BATCH]
        for attempt in range(attempts):
            try:
                r = client.embeddings.create(model="hackathon-embedding", input=batch)
                break
            except openai.RateLimitError as e:
                # Quota 429s carry x-ratelimit-reset; a busy-server 429 carries nothing, so wait a few seconds.
                wait = int(e.response.headers.get("x-ratelimit-reset", 3)) + random.uniform(0, 1)
                print(f"429 on batch starting at {start}; retrying in {wait:.0f}s")
                time.sleep(wait)
        else:
            raise RuntimeError("Embedding failed after repeated 429 responses")
        vectors.extend(item.embedding for item in r.data)
    return np.array(vectors, dtype=np.float32)

sample = embed([f"sentence {i}" for i in range(20)])
print("Shape:", sample.shape)

## Step 4: Search by meaning

Cosine similarity measures how close two vectors point. After normalizing each vector to length 1, the dot product is the cosine similarity: 1.0 means identical direction, 0 means unrelated.

The corpus below is a set of event notes. Embed them once, then score any question against all of them in one matrix multiplication.

In [ ]:
corpus = [
    "The hackathon starts Saturday at 9 AM in the main hall.",
    "Teams may have up to five members.",
    "Submissions are due Sunday at noon through the Devpost page.",
    "Breakfast and lunch are provided both days; dinner is on your own.",
    "Each team receives one API key for the inference endpoint.",
    "The chat model accepts up to about 30,000 tokens of input.",
    "Image generation is limited to six requests per minute per team.",
    "Judging criteria are technical difficulty, design, and usefulness.",
    "Parking is free in the garage on Fifth Street with a validated ticket.",
    "The Wi-Fi network is HackMidwest and the password is printed on your badge.",
    "Mentors wear green lanyards and rotate between tables.",
    "Prizes are announced at the closing ceremony on Sunday at 4 PM.",
]

doc_vectors = embed(corpus)
doc_vectors /= np.linalg.norm(doc_vectors, axis=1, keepdims=True)  # Unit length, so dot product = cosine.

def search(query, k=3):
    q = embed([query])[0]
    q /= np.linalg.norm(q)
    scores = doc_vectors @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), corpus[i]) for i in top]

for score, text in search("When do we have to hand in our project?"):
    print(f"{score:.3f}  {text}")

Notice the top match shares no words with the question. "Hand in our project" found "submissions are due." That is the difference between semantic search and keyword search.

In [ ]:
for score, text in search("Is there food?"):
    print(f"{score:.3f}  {text}")

## Step 5: Ground the chat model in your notes

Ask the chat model about the event directly and it has nothing to go on. It was trained on the public internet, not on your organizers' notes.

In [ ]:
direct = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Where can I park at HackMidwest? One sentence."}],
    max_tokens=60,
)
print(direct.choices[0].message.content)

Retrieval-augmented generation fixes that in three steps: search for the most relevant notes, put them in the prompt, and tell the model to answer only from them. The notes are the facts; the model does the reading and writing.

In [ ]:
def answer(question, k=3):
    hits = search(question, k)
    context = "\n".join(f"- {text}" for _, text in hits)
    prompt = (
        "Answer the question using only the notes below. "
        "If the notes do not contain the answer, say you do not know.\n\n"
        f"Notes:\n{context}\n\nQuestion: {question}"
    )
    r = client.chat.completions.create(
        model="hackathon-chat",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120,
    )
    return r.choices[0].message.content.strip(), hits

reply, hits = answer("Where can I park at HackMidwest?")
print(reply)
print("\nRetrieved notes:")
for score, text in hits:
    print(f"  {score:.3f}  {text}")

In [ ]:
reply, hits = answer("What is the prize money?")
print(reply)

The second question has no answer in the notes, and the instruction in the prompt makes the model say so instead of inventing one. Keep that instruction in every retrieval prompt you write.

## Things to know

- **16 inputs per request.** Use the `embed` helper or write your own batching.
- **8,192 tokens per request in total.** Inputs longer than the model's limit are truncated silently. Split long documents into chunks of a few hundred words before embedding them.
- **Same vectors every time.** Embeddings are deterministic, so embed your documents once and store the vectors. Only the query needs a request at search time.
- **Scale.** NumPy handles thousands of vectors in memory. For more, use a vector database that accepts 1,024-dimension vectors.

## Validation

In [ ]:
score, text = search("How many people can be on a team?", k=1)[0]
assert len(vector) == 1024
assert "five members" in text, text
print(f"Top match ({score:.3f}): {text}")
print("Lab 2 complete.")

## Clean up

Nothing to clean up. The vectors live only in this notebook's memory.

## Next steps

Lab 3 covers image generation: sizes, seeds, the per-team quota, and how to handle a busy queue.